In [108]:
import pandas as pd
import nltk
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet as wn

nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [109]:
dataset = pd.read_csv("DatasetDefinizioni.csv")
print(dataset.dtypes)

Risposta numero                                         int64
(AG) Definizione del concetto ASTRATTO e GENERICO         str
(AS) Definizione del concetto ASTRATTO e SPECIFICO        str
(CG) Definizione del concetto CONCRETO e GENERICO         str
(CS) Definizione del concetto CONCRETO e SPECIFICO        str
(CM) Definizione del concetto misterioso (complesso)      str
dtype: object


In [110]:
df = dataset.drop(columns=['Risposta numero'])

df.rename(columns={
    "(AG) Definizione del concetto ASTRATTO e GENERICO": "AG",
    "(AS) Definizione del concetto ASTRATTO e SPECIFICO": "AS",
    "(CG) Definizione del concetto CONCRETO e GENERICO": "CG",
    "(CS) Definizione del concetto CONCRETO e SPECIFICO": "CS",
    "(CM) Definizione del concetto misterioso (complesso)": "CM"
}, inplace=True)

print(df.dtypes)

df = df.apply(lambda col: col.str.replace(r'\xa0', ' ', regex=True)
                               .str.replace(r'\s+', ' ', regex=True)
                               .str.strip()
                               .str.lower())
df

AG    str
AS    str
CG    str
CS    str
CM    str
dtype: object


,AG,AS,CG,CS,CM
0,art composed of melodious sounds produced by v...,philosophical thought aimed at explaining idea...,"a plant with roots, trunk, branches and foliag...",kitchen tool used to infuse tea leaves in boil...,a feeling of discomfort generated by embarrass...
1,a set of sounds made simultaneously by people ...,branch of philosophy that focuses on defining ...,"plant, with different types and generic size, ...",metal container usually used for heating gener...,abstract mental content and result of mental t...
2,the group of every possibile musical notes com...,common sentiment according with every human be...,high and solid kind of plant composed by a tru...,small container for liquids used to pour hot b...,interactive product of the human intellect and...
3,sound created by one or more musical instruments,philosophy or science that concern behaviors t...,living being that lives in the ground and is c...,tool used to make tea,feeling of discomfort during which one would l...
4,"a mixture of sounds, that can be made of instr...",etics: philosophy’s branch that studies every ...,"plant made by a tunk, branches and leaves",recipient which has a pot-bellied shape with a...,NaN
5,an organizations of sounds that can be made th...,the philosophy that studies what are the right...,a living organism that is rooted in the ground...,a container that can be used to store liquid t...,branches and leaves. these organisms gain thei...
6,sequence of sounds with the aim of triggering ...,branch of philosophy that aims to ask whether ...,woody plant with branches and leaves,"container, often made of ceramic, which is int...",engineering system that has the purpose of cap...
7,sequence of sounds capable of arousing emotion...,moral principles learned by the society to whi...,"type of plant composed of roots, trunk, branch...","container, normally made of ceramic, used to h...",typical japanese dish consisting mainly of ric...
8,set of sounds and harmonies that are heard,set of conventionally shared unwritten rules,"organic component born from a seed, composed o...",container useful for holding and pouring tee,object used to improve vision through the use ...
9,danger is a condition in which there is a poss...,a heuristic is a mental strategy that helps pe...,a tool is an object used to perform practical ...,a microscope is a instrument used to observe o...,it is an instrument used to identify direction...


In [111]:
def get_synsets_from_definition(definition):
    words = word_tokenize(definition.lower())
    synsets = []

    for w in words:
        synsets.extend(wn.synsets(w))

    return synsets

def get_best_synset(definition, lemma):
    tokens = word_tokenize(definition.lower())
    return lesk(tokens, lemma)

data = {}

min_len = min(len(df[col].dropna()) for col in df.columns)

for column in df.columns:
    definitions = df[column].dropna().tolist()[:min_len]

    column_synsets = []

    for definition in definitions:
        tokens = word_tokenize(definition.lower())

        synsets_per_definition = [
            lesk(tokens, word).name()
            for word in tokens
            if lesk(tokens, word) is not None
        ]

        column_synsets.append(synsets_per_definition)

    data[column] = column_synsets

df_synsets = pd.DataFrame(data)
df_synsets.to_csv("DatasetDefinizioni_Synsets.csv", index=False)


df_synsets = df_synsets.drop(columns=['CM'])

df_synsets

,AG,AS,CG,CS
0,"[art.n.03, composed.a.01, tuneful.a.01, sound....","[philosophic.a.01, think.v.03, aim.v.01, astat...","[angstrom.n.01, plant.v.05, root.n.01, trunk.n...","[kitchen.n.01, tool.n.01, use.v.04, steep.v.02..."
1,"[angstrom.n.01, set.n.01, sound.v.04, make.v.1...","[arm.n.02, philosophy.n.02, focus.n.01, on.a.0...","[plant.n.01, different.a.01, type.v.01, generi...","[metallic_element.n.01, container.n.01, normal..."
2,"[group.n.02, every.s.01, musical.n.01, note.n....","[common.s.03, sentiment.n.01, accord.v.02, eve...","[high.a.04, solid.n.02, kind.n.01, plant.n.02,...","[small.n.02, container.n.01, liquid.n.02, use...."
3,"[phone.n.02, make.v.03, by.r.01, one.s.01, ore...","[doctrine.n.01, oregon.n.01, science.n.01, con...","[animation.n.01, organism.n.01, life.n.03, ind...","[creature.n.03, use.v.01, make.v.03, tea.n.05]"
4,"[deoxyadenosine_monophosphate.n.01, mixture.n....","[philosophy.n.02, randomness.n.01, branch.n.02...","[plant.n.02, construct.v.01, by.r.01, deoxyade...","[recipient.n.01, hour_angle.n.02, deoxyadenosi..."
5,"[associate_in_nursing.n.01, organization.n.04,...","[philosophy.n.02, study.n.02, be.v.01, right.n...","[deoxyadenosine_monophosphate.n.01, life.n.02,...","[angstrom.n.01, container.n.01, can.n.02, be.v..."
6,"[succession.n.03, fathom.v.02, aim.n.03, trip....","[branch.n.01, doctrine.n.01, purpose.n.01, ask...","[woody.a.01, plant.n.01, ramify.v.02, leaf.n.01]","[container.n.01, frequently.r.01, make.v.14, c..."
7,"[succession.n.03, sound.n.02, capable.a.03, ar...","[moral.a.01, principle.n.01, memorize.v.01, by...","[type.n.01, plant.n.02, composed.a.01, root.n....","[container.n.01, normally.r.01, cook.v.02, cer..."
8,"[set.n.01, sound.n.02, harmony.n.02, are.n.01,...","[set.n.01, conventionally.r.01, partake.v.02, ...","[organic.n.01, component.n.03, bear.v.11, deox...","[container.n.01, utilitarian.s.01, retain.v.03..."
9,"[danger.n.01, be.v.01, deoxyadenosine_monophos...","[deoxyadenosine_monophosphate.n.01, heuristic....","[angstrom.n.01, creature.n.03, be.v.01, associ...","[angstrom.n.01, microscope.n.01, be.v.01, angs..."


In [112]:
records = []

word_expected = {
    "AG": "music",
    "AS": "ethics",
    "CG": "tree",
    "CS": "teapot",
}

for i in range(len(df)):
    row = {}

    for column in word_expected:
        definition = df[column].iloc[i]
        # Disambiguate the target word using the current definition as context.
        synset = lesk(word_tokenize(definition), word_expected[column])

        row[column] = synset.name() if synset else None

    records.append(row)

df_results = pd.DataFrame(records)
df_results

,AG,AS,CG,CS
0,music.n.04,ethical_motive.n.01,tree.n.01,teapot.n.01
1,music.n.04,ethical_motive.n.01,tree.n.01,teapot.n.01
2,music.n.01,ethical_motive.n.01,tree.n.01,teapot.n.01
3,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
4,music.n.01,ethic.n.01,tree.n.01,teapot.n.01
5,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
6,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
7,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
8,music.n.01,ethics.n.02,tree.n.01,teapot.n.01
9,music.n.01,ethic.n.01,corner.v.02,teapot.n.01
